# Step 1 · The physics and the data
### Burgers' equation shocks: from a PINN to a Fourier Neural Operator

This is notebook 1 of 4 in the class showcase:

| Step | Notebook | What it shows |
|---|---|---|
| **1** | **Physics and data** (this one) | The equation, an exact solution, a trusted solver, and the dataset |
| 2 | PINN | A network that learns **one** solution from the physics alone, with no data |
| 3 | FNO | A network that learns the **solution operator**: any initial condition in, the whole evolution out, in milliseconds |
| 4 | PINO (bonus) | The FNO trained with the PINN's physics loss as well |

Everything later depends on this notebook. If the reference data is wrong, every error plot in Steps 2–4 is meaningless. So we build the solver, **check it against an exact solution**, and only then generate data.

This notebook only needs `numpy`, `scipy` and `matplotlib`. PyTorch comes in at Step 2.

## 1. The equation, and why it makes shocks

Viscous Burgers' equation on a periodic domain $x \in [-1, 1)$, $t \in [0, 1]$:

$$
\underbrace{u_t}_{\text{change in time}} \;+\; \underbrace{u\,u_x}_{\text{self-advection}} \;=\; \underbrace{\nu\, u_{xx}}_{\text{viscosity}}, \qquad \nu = \frac{0.01}{\pi}
$$

Think of $u$ as a velocity, and read the equation one term at a time.

- **$u\,u_x$: each bit of fluid carries itself along at its own speed.** Fast parts catch up with slow parts ahead of them, so the wave front steepens.
- **$\nu\,u_{xx}$: viscosity smooths things out.** It's the heat equation term from the booster course (Module 8).
- **A shock** is the standoff between the two: steepening wins until the front is so thin that viscosity can hold it, about $2\nu/|u| \approx 0.006$ wide here.

**When does it break?** Without viscosity, a point starting at $x_0$ travels at constant speed $u_0(x_0)$, so it sits at $x = x_0 + u_0(x_0)\,t$. Two such paths first collide when $1 + u_0'(x_0)\,t = 0$. For our classic case $u_0 = -\sin(\pi x)$, the steepest descent is $u_0'(0) = -\pi$, so

$$ t_b = \frac{1}{\pi} \approx 0.318. $$

Watch for a shock at $x = 0$ shortly after $t \approx 0.32$.

**Why an aerospace engineer should care.** Burgers' equation is the 1-D toy model of shock formation in compressible flow. It's the same physics as the two shocks on the Geo-FNO transonic airfoil. It is also the standard test problem in *both* papers we're building on: the original PINN paper (Raissi, Perdikaris & Karniadakis, *J. Comput. Phys.* 378, 2019) used exactly this case with $\nu = 0.01/\pi$, and the FNO paper (Li et al., ICLR 2021) used Burgers' equation as its first benchmark.

In [ ]:
import time, os
import numpy as np
from scipy import fft as sfft
import matplotlib.pyplot as plt

NU = 0.01 / np.pi          # viscosity (Raissi et al. 2019 benchmark value)
T_END = 1.0
os.makedirs("data", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# Heads-up for Step 2: check whether PyTorch + CUDA are ready (not needed in this notebook).
try:
    import torch
    print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}",
          f"({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "")
except ImportError:
    print("PyTorch not installed yet. You'll need it for Step 2: https://pytorch.org/get-started/locally/")

## 2. An exact solution, to check everything else against

Here's a small miracle. The **Cole–Hopf transform** $u = -2\nu\,\varphi_x/\varphi$ turns nonlinear Burgers' equation into the *linear heat equation* $\varphi_t = \nu\,\varphi_{xx}$. We know how to solve that exactly: convolve with a Gaussian. Working it through for $u_0 = -\sin(\pi x)$ gives

$$
u(x,t) \;=\; -\,\frac{\displaystyle\int_{-\infty}^{\infty} \sin\!\big(\pi(x-\eta)\big)\, f(x-\eta)\, e^{-\eta^2/4\nu t}\, d\eta}
{\displaystyle\int_{-\infty}^{\infty} f(x-\eta)\, e^{-\eta^2/4\nu t}\, d\eta},
\qquad f(y) = e^{-\cos(\pi y)/(2\pi\nu)} .
$$

That's a convolution again: the same idea as Module 6 of the booster course.

**The numerical catch:** with $\nu = 0.01/\pi$, $f$ swings between $e^{-50}$ and $e^{+50}$. We work with log-weights and subtract the maximum before exponentiating (the log-sum-exp trick), so nothing overflows.

In [ ]:
def burgers_exact_sine(x, t, nu=NU, n_eta=40001, L=3.0, chunk=128):
    # Exact Cole-Hopf solution for u0 = -sin(pi x). Works on any array of x at a single time t.
    x = np.asarray(x, dtype=float)
    if t == 0:
        return -np.sin(np.pi * x)
    eta = np.linspace(-L, L, n_eta)
    out = np.empty_like(x)
    for i in range(0, len(x), chunk):                      # chunk over x to keep memory small
        X = x[i:i+chunk, None] - eta[None, :]
        logw = -np.cos(np.pi * X) / (2*np.pi*nu) - eta[None, :]**2 / (4*nu*t)
        logw -= logw.max(axis=1, keepdims=True)           # log-sum-exp trick
        w = np.exp(logw)
        out[i:i+chunk] = -(np.sin(np.pi * X) * w).sum(1) / w.sum(1)
    return out

## 3. A fast solver for *any* initial condition

The exact formula only works for $-\sin(\pi x)$. To build a dataset for the FNO we need a solver that takes any starting profile. We use a **pseudo-spectral** method. The ideas build up like this:

1. **Derivatives in Fourier space** (booster Module 7). Write $u = \sum_k \hat u_k e^{ikx}$. Then $u_x \leftrightarrow ik\,\hat u_k$ and $u_{xx} \leftrightarrow -k^2 \hat u_k$. On our length-2 domain, $k = \pi n$.
2. **The viscous term exactly** (Module 8). Alone, $u_t = \nu u_{xx}$ just multiplies each mode by $e^{-\nu k^2 \Delta t}$ per step. No stability limit, no error.
3. **The nonlinear term explicitly.** Write $u u_x = \tfrac12 (u^2)_x$. Compute $u^2$ on the grid, transform, and multiply by $ik/2$.
4. **Combine them carefully: ETDRK4.** This is a 4th-order "exponential time-differencing" Runge–Kutta scheme (Cox & Matthews 2002), using the stable coefficient formulas of Kassam & Trefethen (2005).
5. **The 2/3 dealiasing rule.** Squaring $u$ creates frequencies the grid can't hold, and they fold back as fake low frequencies. Zeroing the top third of the modes prevents that.

The solver handles a whole **batch** of initial conditions at once, so generating 1,200 samples is one big array operation.

In [ ]:
def burgers_solve(u0, T=T_END, n_out=51, dt=4e-4, nu=NU):
    # ETDRK4 pseudo-spectral solver for u_t + u u_x = nu u_xx on periodic [-1, 1).
    # u0: (B, N) batch of initial conditions on x_j = -1 + 2j/N.
    # Returns U: (B, n_out, N), snapshots at t = linspace(0, T, n_out).
    u0 = np.atleast_2d(u0).astype(float)
    B, N = u0.shape
    k = np.pi * sfft.rfftfreq(N, d=1.0/N)                 # wavenumbers pi*n
    Lk = -nu * k**2                                        # linear (viscous) operator, diagonal in Fourier space
    E, E2 = np.exp(dt*Lk), np.exp(dt*Lk/2)

    # ETDRK4 coefficients via contour integrals (Kassam & Trefethen 2005), avoids 0/0 at small k
    M = 32
    r = np.exp(1j*np.pi*(np.arange(1, M+1) - 0.5)/M)
    LR = dt*Lk[:, None] + r[None, :]
    Q  = dt*np.real(np.mean((np.exp(LR/2) - 1)/LR, axis=1))
    f1 = dt*np.real(np.mean((-4 - LR + np.exp(LR)*(4 - 3*LR + LR**2))/LR**3, axis=1))
    f2 = dt*np.real(np.mean((2 + LR + np.exp(LR)*(-2 + LR))/LR**3, axis=1))
    f3 = dt*np.real(np.mean((-4 - 3*LR - LR**2 + np.exp(LR)*(4 - LR))/LR**3, axis=1))

    dealias = (np.arange(len(k)) < (2*(N//2))//3).astype(float)   # 2/3 rule
    g = -0.5j * k * dealias
    def nonlinear(vh):                                     # Fourier transform of -(u^2/2)_x
        u = sfft.irfft(vh, n=N, workers=-1)
        return g * sfft.rfft(u*u, workers=-1)

    steps = int(round(T/dt))
    every = steps // (n_out - 1)
    assert every * (n_out - 1) == steps, "choose dt so that T/dt is a multiple of n_out-1"
    vh = sfft.rfft(u0, workers=-1)
    out = [u0.copy()]
    for s in range(1, steps + 1):
        Nv = nonlinear(vh)
        a  = E2*vh + Q*Nv;           Na = nonlinear(a)
        b  = E2*vh + Q*Na;           Nb = nonlinear(b)
        c  = E2*a  + Q*(2*Nb - Nv);  Nc = nonlinear(c)
        vh = E*vh + Nv*f1 + 2*(Na + Nb)*f2 + Nc*f3
        if s % every == 0:
            out.append(sfft.irfft(vh, n=N, workers=-1))
    return np.stack(out, axis=1)

## 4. Trust, but verify: solver vs. exact solution

Run the classic case through the solver at three resolutions and compare with Cole–Hopf. With spectral accuracy we expect the error to fall *dramatically* (not just by a factor of 2) as $N$ doubles.

In [ ]:
t_check = [0.25, 0.5, 0.75, 1.0]
print(f"{'N':>6} {'time [s]':>9}   " + "   ".join(f"t={t:<4}" for t in t_check))
for N, dt in [(512, 5e-4), (1024, 5e-4), (2048, 4e-4)]:
    x = -1 + 2*np.arange(N)/N
    t0 = time.time()
    U = burgers_solve(-np.sin(np.pi*x), n_out=5, dt=dt)[0]
    el = time.time() - t0
    errs = [np.linalg.norm(U[i+1] - burgers_exact_sine(x, t)) / np.linalg.norm(burgers_exact_sine(x, t))
            for i, t in enumerate(t_check)]
    print(f"{N:>6} {el:>9.2f}   " + "   ".join(f"{e:.1e}" for e in errs))

**What you should see:** at $N = 2048$ the relative error is around $10^{-10}$, and it shrinks by orders of magnitude each time $N$ doubles. That's spectral convergence. At low $N$ the error is worst *after* the shock forms ($t \ge 0.5$), because the shock is the thinnest feature on the grid.

So we solve every sample at $N = 2048$ and then **subsample** the grid to store it (every 8th point gives 256 points). Subsampling grid values of an accurate solution loses no accuracy at the stored points.

*Try this:* set `NU = 0.001` at the top and rerun. How much resolution does the solver need now?

In [ ]:
N_REF = 2048
x_ref = -1 + 2*np.arange(N_REF)/N_REF
U_classic = burgers_solve(-np.sin(np.pi*x_ref), n_out=101, dt=4e-4)[0]     # (101, 2048), t = 0, 0.01, ..., 1
t_classic = np.linspace(0, 1, 101)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
im = ax[0].imshow(U_classic.T, extent=[0, 1, -1, 1], origin="lower", aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
ax[0].axvline(1/np.pi, color="k", ls="--", lw=1)
ax[0].text(1/np.pi + 0.01, 0.85, r"$t_b = 1/\pi$", fontsize=11)
ax[0].set(xlabel="t", ylabel="x", title=r"$u(x,t)$ for $u_0=-\sin(\pi x)$")
plt.colorbar(im, ax=ax[0], label="u")
for t in [0, 0.25, 0.5, 0.75, 1.0]:
    ax[1].plot(x_ref, U_classic[int(round(t*100))], label=f"t = {t:.2f}")
ax[1].set(xlabel="x", ylabel="u", title="Snapshots: the wave steepens into a shock at x = 0")
ax[1].legend()
plt.tight_layout(); plt.show()

## 5. A dataset of random initial conditions (for the FNO)

The PINN in Step 2 needs **no data**, only the equation. The FNO in Step 3 learns from examples, so we need lots of *(initial condition → full evolution)* pairs.

We draw random initial conditions from a **Gaussian random field**, the same recipe as the FNO paper: $u_0 \sim \mathcal N\big(0,\;625\,(-\Delta + 25 I)^{-2}\big)$. In practice, that's a random Fourier series whose mode-$n$ amplitude has variance $625/((\pi n)^2 + 25)^2$. Big waves are common, small wiggles are rare. We then rescale each sample so its peak speed is between 0.5 and 1, the same range as the classic $-\sin(\pi x)$ case. That keeps the PINN's test case inside the FNO's training distribution.

**Dataset layout**

| Split | Samples | Stored grid | Snapshots | Purpose |
|---|---|---|---|---|
| train | 1,000 | 256 points | 51 ($\Delta t = 0.02$) | Fit the FNO |
| test | 200 | 1,024 points | 51 | Error, plus zero-shot super-resolution (train at 256, test at 1,024) |
| classic | 1 | 1,024 points | 101 ($\Delta t = 0.01$) | Head-to-head PINN vs FNO comparison |

In [ ]:
def random_ic(B, N, rng, amp=(0.5, 1.0)):
    # Gaussian random field initial conditions on periodic [-1, 1), rescaled to peak |u0| in amp.
    x = -1 + 2*np.arange(N)/N
    n = np.arange(1, N//2)
    std = np.sqrt(625.0 / ((np.pi*n)**2 + 25.0)**2)
    a = rng.standard_normal((B, len(n))) * std
    b = rng.standard_normal((B, len(n))) * std
    u0 = a @ np.cos(np.pi*np.outer(n, x)) + b @ np.sin(np.pi*np.outer(n, x))
    u0 *= rng.uniform(*amp, size=(B, 1)) / np.abs(u0).max(axis=1, keepdims=True)
    return u0

N_TRAIN, N_TEST, BATCH = 1000, 200, 200     # lower these for a quick dry run
N_OUT = 51

def make_split(n_samples, store_every, seed):
    rng = np.random.default_rng(seed)
    chunks = []
    for start in range(0, n_samples, BATCH):
        B = min(BATCH, n_samples - start)
        t0 = time.time()
        U = burgers_solve(random_ic(B, N_REF, rng), n_out=N_OUT, dt=4e-4)
        chunks.append(U[:, :, ::store_every].astype(np.float32))
        print(f"  samples {start:4d}-{start+B-1:4d} solved in {time.time()-t0:5.1f}s")
    return np.concatenate(chunks)

print("Train set (stored at 256 points):")
U_train = make_split(N_TRAIN, store_every=8, seed=0)
print("Test set (stored at 1024 points):")
U_test = make_split(N_TEST, store_every=2, seed=1)

In [ ]:
t_grid = np.linspace(0, T_END, N_OUT).astype(np.float32)
np.savez_compressed("data/burgers_train.npz", u=U_train, t=t_grid, x=(-1 + 2*np.arange(256)/256).astype(np.float32), nu=NU)
np.savez_compressed("data/burgers_test.npz",  u=U_test,  t=t_grid, x=(-1 + 2*np.arange(1024)/1024).astype(np.float32), nu=NU)
np.savez_compressed("data/burgers_classic.npz", u=U_classic[:, ::2].astype(np.float32), t=t_classic.astype(np.float32),
                    x=x_ref[::2].astype(np.float32), nu=NU)

for name, U in [("train", U_train), ("test", U_test), ("classic", U_classic[:, ::2])]:
    print(f"{name:8s} shape={U.shape}  max|u|={np.abs(U).max():.3f}  finite={np.isfinite(U).all()}")
print("Saved to ./data/")

## 6. Look at what we made

Each colour below is one random initial condition (dashed) and its state at $t = 1$ (solid). Several bumps steepen into several shocks: a sawtooth. The FNO has to learn to produce these from the dashed curves alone.

In [ ]:
x256 = -1 + 2*np.arange(256)/256
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for j in range(4):
    ax[0].plot(x256, U_train[j, 0], "--", color=f"C{j}", alpha=0.6)
    ax[0].plot(x256, U_train[j, -1], color=f"C{j}")
ax[0].set(xlabel="x", ylabel="u", title="Random initial conditions (dashed) and t = 1 (solid)")
im = ax[1].imshow(U_train[0].T, extent=[0, 1, -1, 1], origin="lower", aspect="auto", cmap="RdBu_r")
ax[1].set(xlabel="t", ylabel="x", title="Training sample 0: shocks forming and merging")
plt.colorbar(im, ax=ax[1], label="u")
plt.tight_layout(); plt.show()

## Recap, and what comes next

- We have an **exact solution** for the classic case and a **spectral solver** that matches it to about $10^{-10}$.
- We have **1,200 solved samples** saved in `./data/`, plus the classic case.

**Step 2: the PINN.** We'll train a small neural network $u_\theta(x,t)$ with *no data at all*. The loss will be the PDE residual $u_t + u u_x - \nu u_{xx}$, computed with automatic differentiation, plus the initial condition. Then we'll see how well it captures the shock at $x = 0$, and how long it takes to learn just this *one* solution.